In [1]:
using Makie
using Colors
using Base.Threads

# Simple fake ray tracing function that creates a gradient
function fake_trace_ray(x, y, width, height, frame_offset=0)
    # Create a moving pattern
    r = (x + frame_offset) / width
    g = y / height
    b = sin(x * 0.1 + frame_offset * 0.1) * 0.5 + 0.5
    return RGB{Float32}(r, g, b)
end

# Single-threaded version
function render_singlethread!(img_observable; update_frequency=10)
    img = img_observable[]
    height, width = size(img)
    
    for y in 1:height
        for x in 1:width
            color = fake_trace_ray(x, y, width, height)
            img[y, x] = color
        end
        
        if y % update_frequency == 0
            notify(img_observable)
            sleep(0.001)
        end
    end
    
    notify(img_observable)
end

# Multi-threaded tile-based version
function render_multithreaded!(img_observable; tile_size=32, update_frequency=5)
    img = img_observable[]
    height, width = size(img)
    
    tiles = [(tx, ty) for ty in 1:tile_size:height for tx in 1:tile_size:width]
    completed = Atomic{Int}(0)
    
    @threads for tile_idx in 1:length(tiles)
        tx, ty = tiles[tile_idx]
        
        for y in ty:min(ty + tile_size - 1, height)
            for x in tx:min(tx + tile_size - 1, width)
                color = fake_trace_ray(x, y, width, height)
                img[y, x] = color
            end
        end
        
        count = atomic_add!(completed, 1)
        if count % update_frequency == 0
            @async notify(img_observable)
        end
    end
    
    notify(img_observable)
end

# Setup visualization
function setup_visualization(width, height)
    img = Observable(zeros(RGB{Float32}, height, width))
    
    fig = Figure(size=(width, height))
    ax = Axis(fig[1, 1], aspect=DataAspect())
    image!(ax, img)
    hidedecorations!(ax)
    display(fig)
    
    return img
end

# Test it!
width, height = 400, 300
img_obs = setup_visualization(width, height)

# Try single-threaded
println("Rendering single-threaded...")
@time render_singlethread!(img_obs, update_frequency=5)

# Reset and try multi-threaded
sleep(1)
img_obs[] = zeros(RGB{Float32}, height, width)
notify(img_obs)

println("\nRendering multi-threaded with $(nthreads()) threads...")
@time render_multithreaded!(img_obs, tile_size=32, update_frequency=5)

ErrorException: No backend available!
Make sure to also `import/using` a backend (GLMakie, CairoMakie, WGLMakie).

If you imported GLMakie, it may have not built correctly.
In that case, try `]build GLMakie` and watch out for any warnings.
